In [1]:
# Test script 3

In [1]:
import os
import xarray as xr
import numpy as np
import xesmf as xe
from utils.utils import adjust_longitude

In [2]:
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8/"
OBS_DIR = "/glade/work/awells/air_quality/O3_obs/"
scenario = "ARISE"
ens_num = 1

# Load data arrays
if scenario == "ARISE":
    dates = "2035-2068"
elif scenario == "SSP245":
    dates = "2020-2068"

osdma8_file = f"OSDMA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
osdma8_path = os.path.join(OSDMA8_DIR, osdma8_file)
# Convert from mol/mol to ppb
osdma8 = xr.open_dataarray(osdma8_path)*10**9

hist_file = "OSDMA8_CESM2_hist_01_1990-2008.nc"
hist_path = os.path.join(OSDMA8_DIR, hist_file)
# Convert from mol/mol to ppb
hist = xr.open_dataarray(hist_path)*10**9

obs_file = "Delang_BME_OSDMA8_1990_2017.nc"
obs_path = os.path.join(OBS_DIR, obs_file)
obs = xr.open_dataset(obs_path)["ozone"]

# Baseline years for fi_2000 and historical
base = slice("1990", "2008")
hist_base = hist.sel(year=base).mean("year")
obs_base = obs.sel(year=base).mean("year")
obs_base = obs_base.rename({'longitude': 'lon', 'latitude': 'lat'})

# Define the higher-resolution grid to match observations (0.1° x 0.1°)
new_lat = obs_base['lat']
new_lon = obs_base['lon']

# Calculate delta
delta_fi = adjust_longitude(osdma8 / hist_base)

In [23]:
osdma8

<xarray.DataArray (year: 34, lat: 192, lon: 288)> Size: 15MB
array([[[32.84760879, 32.84760877, 32.84760877, ..., 32.84760879,
         32.84760879, 32.84760879],
        [32.9077986 , 32.90815895, 32.90850018, ..., 32.90660577,
         32.90701896, 32.90742655],
        [32.905274  , 32.9058487 , 32.9063038 , ..., 32.90305459,
         32.90383708, 32.90460258],
        ...,
        [45.5424551 , 45.54082329, 45.53902741, ..., 45.5791855 ,
         45.57351233, 45.55657085],
        [45.24227313, 45.27362663, 45.27397852, ..., 45.25982562,
         45.25040326, 45.25504523],
        [44.62838612, 44.62841813, 44.62844665, ..., 44.62827307,
         44.62831343, 44.62835236]],

       [[34.40440235, 34.40440235, 34.40440235, ..., 34.40440235,
         34.40440235, 34.40440235],
        [34.51074463, 34.51125444, 34.51175668, ..., 34.50923955,
         34.50973757, 34.51022539],
        [34.60425531, 34.6056561 , 34.60697953, ..., 34.59963289,
         34.60133824, 34.60281725],
...
        [46.70568558, 46.69873967, 46.68351733, ..., 46.76053527,
         46.74268215, 46.7179062 ],
        [46.30021637, 46.31263132, 46.31657639, ..., 46.32617299,
         46.31673733, 46.31768336],
        [45.25489523, 45.25490443, 45.25486247, ..., 45.2551435 ,
         45.2549213 , 45.2548966 ]],

       [[30.34759058, 30.34759058, 30.34759056, ..., 30.34759058,
         30.34759058, 30.34759058],
        [30.42874503, 30.42904306, 30.4293158 , ..., 30.42754154,
         30.42797259, 30.42838578],
        [30.46881514, 30.47043989, 30.47228048, ..., 30.46424379,
         30.46570539, 30.46720344],
        ...,
        [46.11351726, 46.10040348, 46.0791808 , ..., 46.15481165,
         46.14286745, 46.12767675],
        [45.67900561, 45.68522911, 45.67437164, ..., 45.71781852,
         45.70083629, 45.70028819],
        [44.58208148, 44.5823295 , 44.58297597, ..., 44.58174263,
         44.58222667, 44.58175725]]])
Coordinates:
  * year     (year) int64 272B 2035 2036 2037 2038 2039 ... 2065 2066 2067 2068
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
    time     (year, lat, lon) object 15MB ...

In [ ]:
def bilinear_interp(in_grid, target_grid):
    regridder = xe.Regridder(in_grid, target_grid, method="bilinear")
    return regridder


# Interpolate to the new grid
regridder = bilinear_interp(delta_fi, obs_base)
ds_delta_fi = regridder(delta_fi)

# Bias correct delta
bc_data = obs_base * ds_delta_fi

In [8]:
test = xe.Regridder(in_grid, out_grid, method="bilinear")

In [11]:
new_delta = test(delta_fi)

/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


In [14]:
# Interpolate to the new grid
ds_delta_fi = delta_fi.interp(lat=new_lat, lon=new_lon,
                              method='linear')

# Bias correct delta
bc_data = obs_base * ds_delta_fi

In [11]:
# === Define resolution and bounds ===
lat_res = 1 / 10  # 0.1 degrees
lon_res = 1 / 10

# Generate full latitude and longitude ranges
lat_full = np.arange(-90 + lat_res / 2, 90, lat_res)
lon_full = np.arange(-180 + lon_res / 2, 180, lon_res)

# Confirm lengths match 0.1 x 0.1 grid
print((f"Longitude points should be 3600: {len(lon_full)}, "
       f"Latitude points should be 1800: {len(lat_full)}"))

# === Create new empty DataArray with NaNs ===
shape = (len(lat_full), len(lon_full), len(bc_data.year))
bc_full = xr.DataArray(
    data=np.full(shape, np.nan),
    coords={"lat": lat_full, "lon": lon_full, "year": bc_data.year.values},
    dims=["lat", "lon", "year"],
    name="OSDMA8 Bias Corrected"
)

Longitude points should be 3600: 3600, Latitude points should be 1800: 1800


In [18]:
# === Add OSDMA8 data to empty DataArray ===
# Adjust indices to match (with small tolerance) e.g., max 0.01° distance
bc_nearest = bc_data.reindex_like(bc_full, method="nearest", tolerance=0.01)

# Add data to bc_full
mask = xr.where(bc_full.isnull(), True, False)
bc_full = xr.where(mask, bc_nearest, bc_full)

In [21]:
bc_full = bc_full.drop_vars("time")
bc_full.to_netcdf("/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/test.nc")